In [20]:
import os

os.makedirs("/content/ml_project", exist_ok=True)
os.chdir("/content/ml_project")

print("Project folder created")
print(os.getcwd())

Project folder created
/content/ml_project


In [21]:
%%writefile app.py

from flask import Flask, request, jsonify
import joblib

app = Flask(__name__)

model = joblib.load("model.pkl")

@app.route("/")
def home():
    return "ML Model API is running!"

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()

    features = data["features"]

    prediction = model.predict([features])[0]

    return jsonify({
        "prediction": int(prediction)
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [22]:
%%writefile requirements.txt

Flask
joblib
scikit-learn
pandas
numpy

Overwriting requirements.txt


In [23]:
%%writefile Dockerfile

FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY model.pkl .

EXPOSE 5000

CMD ["python", "app.py"]

Overwriting Dockerfile


In [24]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Load dataset
df = pd.read_csv("/content/data.csv")

# Remove unnecessary columns
df = df.drop(["id", "Unnamed: 32"], axis=1, errors="ignore")

# Convert diagnosis into numbers
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

# Separate input and output
X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Handle missing values
imputer = SimpleImputer(strategy="mean")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Accuracy
accuracy = model.score(X_test, y_test)
print("Model trained successfully")
print("Accuracy:", accuracy)

# Save trained model
joblib.dump(model, "/content/model.pkl")

print("model.pkl created successfully")

Model trained successfully
Accuracy: 0.956140350877193
model.pkl created successfully


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
import shutil

shutil.copy("/content/model.pkl", "/content/ml_project/model.pkl")

print("model.pkl copied successfully")

model.pkl copied successfully


In [26]:
import os

print(os.listdir("/content/ml_project"))

['requirements.txt', 'Dockerfile', 'app.py', 'model.pkl']


In [27]:
%cd /content/ml_project

!ls -lh

/content
total 16K
-rw-r--r-- 1 root root  482 Sep  3 17:26 app.py
-rw-r--r-- 1 root root  185 Sep  3 17:26 Dockerfile
-rw-r--r-- 1 root root 1.1K Sep  3 17:26 model.pkl
-rw-r--r-- 1 root root   40 Sep  3 17:26 requirements.txt


In [28]:
!apt-get update -qq
!apt-get install -y docker.io -qq

!docker --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Docker version 29.1.3, build 29.1.3-0ubuntu3~22.04.2


In [29]:
!pkill dockerd || true
!pkill containerd || true
!rm -f /var/run/docker.pid

!dockerd \
  --iptables=false \
  --bridge=none \
  --storage-driver=vfs \
  --exec-opt native.cgroupdriver=cgroupfs \
  --cgroup-parent="" \
  > /tmp/dockerd.log 2>&1 &

!sleep 10

!docker info | grep -E "Server Version|Storage Driver|Cgroup Driver"

 Server Version: 29.1.3
 Storage Driver: vfs
 Cgroup Driver: cgroupfs


In [30]:
%cd /content/ml_project

/content


In [31]:
!docker build -t ml-flask-app .

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/


Step 1/8 : FROM python:3.10-slim
 ---> db825c749364
Step 2/8 : WORKDIR /app
 ---> Using cache
 ---> 547cfce5a934
Step 3/8 : COPY requirements.txt .
 ---> Using cache
 ---> e748a2f2b4e8
Step 4/8 : RUN pip install --no-cache-dir -r requirements.txt
 ---> Running in 33f34179b148
failed to create task for container: failed to create shim task: OCI runtime create failed: runc create failed: unable to start container process: unable to apply cgroup configuration: mkdir /sys/fs/cgroup/docker: read-only file system


In [32]:
!docker build -t ml-flask-app .

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/


Step 1/8 : FROM python:3.10-slim
 ---> db825c749364
Step 2/8 : WORKDIR /app
 ---> Using cache
 ---> 547cfce5a934
Step 3/8 : COPY requirements.txt .
 ---> Using cache
 ---> e748a2f2b4e8
Step 4/8 : RUN pip install --no-cache-dir -r requirements.txt
 ---> Running in 412fd9e5762a
failed to create task for container: failed to create shim task: OCI runtime create failed: runc create failed: unable to start container process: unable to apply cgroup configuration: mkdir /sys/fs/cgroup/docker: read-only file system


In [33]:
!docker run -d -p 5000:5000 --name ml-container ml-flask-app

Unable to find image 'ml-flask-app:latest' locally
docker: Error response from daemon: pull access denied for ml-flask-app, repository does not exist or may require 'docker login': denied: requested access to the resource is denied

Run 'docker run --help' for more information


In [34]:
!docker ps

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES
